In [ ]:
import sys
print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Python executable: /usr/bin/python3


In [3]:
!pip install -q uv

!uv python install 3.12

!uv venv --python 3.12 /content/mage-env

!uv pip install --python /content/mage-env/bin/python mage-ai==0.9.80

Installed Python 3.12.16 in 1.02s
 + cpython-3.12.16-linux-x86_64-gnu (python3.12)
Using CPython 3.12.16
Creating virtual environment at: mage-env
Activate with: source mage-env/bin/activate
Using Python 3.12.16 environment at: mage-env
Resolved 182 packages in 1.48s
Prepared 182 packages in 6.96s
Installed 182 packages in 401ms
 + aiofiles==22.1.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.10.0
 + aiosignal==1.4.0
 + alembic==1.14.1
 + altair==4.2.2
 + annotated-types==0.8.0
 + anyio==4.14.2
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==26.2.0
 + arrow==1.4.0
 + asttokens==3.0.2
 + async-lru==2.4.0
 + attrs==26.1.0
 + babel==2.18.0
 + backcall==0.2.0
 + bcrypt==4.0.1
 + beautifulsoup4==4.15.0
 + bleach==6.4.0
 + cachetools==8.0.0
 + certifi==2026.10.23
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.1.8
 + cloudpickle==3.1.2
 + colorama==0.4.6
 + comm==0.2.3
 + cron-converter==1.2.1
 + cryptography==42.0.4
 + dask==2026.8.0
 + datadog==0.44.0
 + debugpy==1.8.22
 + decorator==5

In [5]:
! /content/mage-env/bin/mage --help

                                                                                
 Usage: mage [OPTIONS] COMMAND [ARGS]...                                        
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --install-completion          Install completion for the current shell.      │
│ --show-completion             Show completion for the current shell, to copy │
│                               it or customize the installation.              │
│ --help                        Show this message and exit.                    │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ init                           Initialize Mage project.                      │
│ start                          Start Mage server and UI.                     │
│ run                       

In [6]:
! /content/mage-env/bin/python -c "import mage_ai; print('Mage import successful')"

Mage import successful


In [7]:
%cd /content
! /content/mage-env/bin/mage init iris_project

/content
Initialized Mage project at /content/iris_project


In [8]:
%cd /content/iris_project

!find . -maxdepth 3 -type f | sort

/content/iris_project
./charts/__init__.py
./custom/__init__.py
./data_exporters/export_titanic_clean.py
./data_exporters/__init__.py
./data_loaders/__init__.py
./data_loaders/load_titanic.py
./dbt/profiles.yml
./extensions/__init__.py
./.gitignore
./__init__.py
./interactions/__init__.py
./io_config.yaml
./metadata.yaml
./pipelines/example_pipeline/__init__.py
./pipelines/example_pipeline/metadata.yaml
./pipelines/__init__.py
./requirements.txt
./scratchpads/__init__.py
./transformers/fill_in_missing_values.py
./transformers/__init__.py
./utils/__init__.py


In [9]:
import os

project = "/content/iris_project"

folders = [
    "pipelines/iris_classifier",
    "data_loaders",
    "transformers",
    "data_exporters"
]

for folder in folders:
    os.makedirs(os.path.join(project, folder), exist_ok=True)

print("Iris pipeline folders created.")

Iris pipeline folders created.


In [10]:
%%writefile /content/iris_project/pipelines/iris_classifier/metadata.yaml
name: iris_classifier
type: python

Writing /content/iris_project/pipelines/iris_classifier/metadata.yaml


In [11]:
%%writefile /content/iris_project/data_loaders/load_iris.py

from sklearn.datasets import load_iris
import pandas as pd


@data_loader
def load_iris_data(*args, **kwargs):
    iris = load_iris(as_frame=True)
    df = iris.frame

    return df


@test
def test_output(output, *args):
    assert output is not None
    assert len(output) == 150
    assert output.shape[1] == 5
    assert "target" in output.columns
    assert output["target"].nunique() == 3

Writing /content/iris_project/data_loaders/load_iris.py


In [13]:
%%writefile /content/iris_project/data_loaders/load_iris.py

from sklearn.datasets import load_iris
import pandas as pd


@data_loader
def load_iris_data(*args, **kwargs):
    iris = load_iris(as_frame=True)
    return iris.frame


@test
def test_output(output, *args):
    assert output is not None
    assert len(output) == 150
    assert output.shape[1] == 5
    assert "target" in output.columns
    assert output["target"].nunique() == 3

Overwriting /content/iris_project/data_loaders/load_iris.py


In [14]:
import sys
sys.path.insert(0, "/content/iris_project")

from sklearn.datasets import load_iris

df = load_iris(as_frame=True).frame

assert len(df) == 150
assert df.shape[1] == 5
assert df["target"].nunique() == 3

print("Dataset shape:", df.shape)
print("Classes:", df["target"].nunique())
print("Dataset validation passed!")

Dataset shape: (150, 5)
Classes: 3
Dataset validation passed!


In [15]:
%%writefile /content/iris_project/transformers/prepare_data.py

@transformer
def prepare_data(df, *args, **kwargs):
    df = df.copy()
    df["split"] = "train"

    test_indices = df.sample(
        frac=0.2,
        random_state=42
    ).index

    df.loc[test_indices, "split"] = "test"

    return df


@test
def test_output(output, *args):
    assert output is not None
    assert len(output) == 150
    assert (output["split"] == "train").sum() == 120
    assert (output["split"] == "test").sum() == 30

Writing /content/iris_project/transformers/prepare_data.py


In [16]:
%%writefile /content/iris_project/transformers/train_model.py

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


@transformer
def train_model(df, *args, **kwargs):
    train_df = df[df["split"] == "train"]
    test_df = df[df["split"] == "test"]

    feature_cols = [
        col for col in df.columns
        if col not in ["target", "split"]
    ]

    X_train = train_df[feature_cols]
    y_train = train_df["target"]

    X_test = test_df[feature_cols]
    y_test = test_df["target"]

    model = DecisionTreeClassifier(max_depth=3, random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    return {
        "model": model,
        "accuracy": accuracy,
        "predictions": predictions,
        "y_test": y_test
    }


@test
def test_output(output, *args):
    assert output is not None
    assert output["model"] is not None
    assert 0 <= output["accuracy"] <= 1

Writing /content/iris_project/transformers/train_model.py


In [17]:
%%writefile /content/iris_project/data_exporters/save_results.py

import pandas as pd


@data_exporter
def save_results(model_output, *args, **kwargs):
    results = pd.DataFrame({
        "actual": model_output["y_test"].values,
        "predicted": model_output["predictions"]
    })

    results.to_csv("iris_predictions.csv", index=False)

    return results


@test
def test_output(output, *args):
    assert output is not None
    assert len(output) == 30
    assert "actual" in output.columns
    assert "predicted" in output.columns

Writing /content/iris_project/data_exporters/save_results.py


In [18]:
%%writefile /content/iris_project/pipelines/iris_classifier/metadata.yaml

blocks:
  - uuid: load_iris
    name: load_iris
    type: data_loader
    downstream_blocks:
      - prepare_data

  - uuid: prepare_data
    name: prepare_data
    type: transformer
    downstream_blocks:
      - train_model

  - uuid: train_model
    name: train_model
    type: transformer
    downstream_blocks:
      - save_results

  - uuid: save_results
    name: save_results
    type: data_exporter
    downstream_blocks: []

name: iris_classifier
type: python

Overwriting /content/iris_project/pipelines/iris_classifier/metadata.yaml


In [19]:
!nohup /content/mage-env/bin/mage start /content/iris_project --host 0.0.0.0 --port 6789 > /content/mage.log 2>&1 &

In [20]:
import time
time.sleep(10)

!tail -30 /content/mage.log

INFO:alembic.runtime.migration:Running upgrade 83a74eca6ea1 -> e7beb59b44f9, Add repo path to pipeline schedule
INFO:alembic.runtime.migration:Running upgrade e7beb59b44f9 -> dfe49d040487, Add condition_failed status to block run status
INFO:alembic.runtime.migration:Running upgrade dfe49d040487 -> 90d978a8aef8, Update unique constraint for secret
INFO:alembic.runtime.migration:Running upgrade 90d978a8aef8 -> e3593cc2191e, Add key uuid to secrets
INFO:alembic.runtime.migration:Running upgrade e3593cc2191e -> 66928ca5b0d7, Add global_data_product_uuid to PipelineSchedule
INFO:alembic.runtime.migration:Running upgrade 66928ca5b0d7 -> 386bcfebd48d, Create Tag and TagAssociation tables
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
INF

In [21]:
!curl -I http://127.0.0.1:6789

HTTP/1.1 405 Method Not Allowed
Server: TornadoServer/6.4.2
Content-Type: text/html; charset=UTF-8
Date: Mon, 16 Nov 2026 12:18:26 GMT
Content-Length: 87



In [22]:
!tail -20 /content/mage.log

INFO:mage_ai.orchestration.pipeline_scheduler_original:Active pipeline runs: []
INFO:mage_ai.server.server:User with owner permission doesn’t exist, creating owner user.
/content/mage-env/lib/python3.12/site-packages/mage_ai/orchestration/db/cache.py:44: SAWarning: Object of type <User> not in session, add operation along 'Role.users' won't proceed (This warning originated from the Session 'autoflush' process, which was invoked automatically in response to a user-initiated operation. Consider using ``no_autoflush`` context manager if this warning happened while initializing objects.)
  results = super().all(*args, **kwargs)
INFO:mage_ai.server.server:OAuth2 application doesn’t exist for frontend, creating OAuth2 application.
INFO:mage_ai.server.server:Initializing block cache.
INFO:mage_ai.server.server:Initializing pipeline cache.
INFO:mage_ai.server.server:Initializing tag cache.
INFO:mage_ai.server.server:Initializing block action object cache.
INFO:mage_ai.orchestration.pipeline_sc

In [23]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.11.0) ...
Setting up cloudflared (2026.11.0) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [24]:
!nohup cloudflared tunnel --url http://localhost:6789 > /content/cloudflared.log 2>&1 &

In [25]:
import time
time.sleep(10)

!grep -o 'https://[^ ]*trycloudflare.com' /content/cloudflared.log | head -1

https://balloon-telling-system-forward.trycloudflare.com


In [26]:
import os

for key in [
    "DEFAULT_OWNER_EMAIL",
    "DEFAULT_OWNER_USERNAME",
    "DEFAULT_OWNER_PASSWORD"
]:
    print(key, "configured:", bool(os.environ.get(key)))

DEFAULT_OWNER_EMAIL configured: False
DEFAULT_OWNER_USERNAME configured: False
DEFAULT_OWNER_PASSWORD configured: False


In [27]:
!pkill -f "mage start" || true

^C


In [28]:
!find /content/iris_project /root/.mage_data /root -maxdepth 3 \
  \( -name "*.db" -o -name "*.sqlite" -o -name "*.sqlite3" \) \
  2>/dev/null | head -30

/root/.mage_data/iris_project/mage-ai.db
/root/.mage_data/content/mage-ai.db
/root/.ipython/profile_default/history.sqlite
/root/.mage_data/iris_project/mage-ai.db
/root/.mage_data/content/mage-ai.db


In [29]:
import sqlite3
import os
from datetime import datetime

paths = [
    "/root/.mage_data/iris_project/mage-ai.db",
    "/root/.mage_data/content/mage-ai.db"
]

for path in paths:
    print("\nDatabase:", path)

    if not os.path.exists(path):
        print("Not found")
        continue

    print("Last modified:",
          datetime.fromtimestamp(os.path.getmtime(path)))

    conn = sqlite3.connect(path)
    tables = [
        row[0] for row in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        )
    ]

    print("Has users table:", "user" in tables or "users" in tables)

    for table in ["user", "users", "pipeline", "pipelines"]:
        if table in tables:
            count = conn.execute(
                f'SELECT COUNT(*) FROM "{table}"'
            ).fetchone()[0]
            print(f"{table}: {count} rows")

    conn.close()


Database: /root/.mage_data/iris_project/mage-ai.db
Last modified: 2026-11-16 12:17:34.334624
Has users table: True
user: 1 rows

Database: /root/.mage_data/content/mage-ai.db
Last modified: 2026-11-16 12:07:23.603442
Has users table: False


In [30]:
import os
import shutil

db = "/root/.mage_data/iris_project/mage-ai.db"

if os.path.exists(db):
    shutil.copy2(db, db + ".backup")
    os.remove(db)

print("Active Mage database backed up and reset.")

Active Mage database backed up and reset.


In [31]:
import os
from getpass import getpass

os.environ["DEFAULT_OWNER_EMAIL"] = input("Enter owner email: ")
os.environ["DEFAULT_OWNER_USERNAME"] = "admin"
os.environ["DEFAULT_OWNER_PASSWORD"] = getpass("Create Mage password: ")

print("Mage owner credentials configured.")

Enter owner email: harshithscrapmail@gmail.com
Create Mage password: ··········
Mage owner credentials configured.


In [32]:
!nohup /content/mage-env/bin/mage start /content/iris_project --host 0.0.0.0 --port 6789 > /content/mage.log 2>&1 &

In [33]:
import time
time.sleep(15)
!tail -25 /content/mage.log

INFO:alembic.runtime.migration:Running upgrade dfe49d040487 -> 90d978a8aef8, Update unique constraint for secret
INFO:alembic.runtime.migration:Running upgrade 90d978a8aef8 -> e3593cc2191e, Add key uuid to secrets
INFO:alembic.runtime.migration:Running upgrade e3593cc2191e -> 66928ca5b0d7, Add global_data_product_uuid to PipelineSchedule
INFO:alembic.runtime.migration:Running upgrade 66928ca5b0d7 -> 386bcfebd48d, Create Tag and TagAssociation tables
INFO:alembic.runtime.migration:Running upgrade 386bcfebd48d -> a38d30f2541c, Add started_at for pipeline run
INFO:alembic.runtime.migration:Running upgrade a38d30f2541c -> 117f8c3b9c42, Add description to pipeline_schedule
INFO:alembic.runtime.migration:Running upgrade 117f8c3b9c42 -> 526c4523d5d5, Add user_id to Roles and more entity columns to Permissions
INFO:alembic.runtime.migration:Running upgrade 526c4523d5d5 -> 9f3b46404196, Create RolePermission table
INFO:alembic.runtime.migration:Running upgrade 9f3b46404196 -> 03135dd2b4be, Add 

In [34]:
%cd /content

! /content/mage-env/bin/mage run iris_project iris_classifier --test

/content
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /content/mage-env/lib/python3.12/site-packages/mage_ai/data_preparation/exec │
│ utors/pipeline_executor.py:82 in execute                                     │
│                                                                              │
│    79 │   │   │   │   )                                                      │
│    80 │   │   # Execute the task based on current context                    │
│    81 │   │   try:                                                           │
│ ❱  82 │   │   │   loop = asyncio.get_running_loop()                          │
│    83 │   │   │   logger.info(f'[PipelineExecutor] Found running loop {loop} │
│    84 │   │   │   # We're in an async context, use create_task               │
│    85 │   │   │   task = asyncio.create_

In [35]:
%%writefile /content/iris_project/pipelines/iris_classifier/metadata.yaml

name: iris_classifier
type: python

blocks:
  - uuid: load_iris
    name: load_iris
    type: data_loader
    upstream_blocks: []
    downstream_blocks:
      - prepare_data

  - uuid: prepare_data
    name: prepare_data
    type: transformer
    upstream_blocks:
      - load_iris
    downstream_blocks:
      - train_model

  - uuid: train_model
    name: train_model
    type: transformer
    upstream_blocks:
      - prepare_data
    downstream_blocks:
      - save_results

  - uuid: save_results
    name: save_results
    type: data_exporter
    upstream_blocks:
      - train_model
    downstream_blocks: []

Overwriting /content/iris_project/pipelines/iris_classifier/metadata.yaml


In [36]:
%cd /content

! /content/mage-env/bin/mage run iris_project iris_classifier --test

/content
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [39]:
! /content/mage-env/bin/mage run iris_project iris_classifier --test

DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [40]:
import pandas as pd

df = pd.read_csv("/content/iris_project/iris_predictions.csv")
display(df)

FileNotFoundError: [Errno 2] No such file or directory: '/content/iris_project/iris_predictions.csv'

In [41]:
!find /content /root/.mage_data -name "iris_predictions.csv" 2>/dev/null

/content/iris_predictions.csv


In [42]:
import pandas as pd

df = pd.read_csv("/content/iris_predictions.csv")
display(df)

print("Total predictions:", len(df))


,actual,predicted
0,{'target': 0},numpy.ndarray
1,{'target': 0},numpy.ndarray
2,{'target': 0},numpy.ndarray
3,{'target': 0},numpy.ndarray
4,{'target': 0},numpy.ndarray
5,{'target': 0},numpy.ndarray
6,{'target': 0},numpy.ndarray
7,{'target': 0},numpy.ndarray
8,{'target': 0},numpy.ndarray
9,{'target': 0},numpy.ndarray


Total predictions: 30


In [43]:
%%writefile /content/iris_project/transformers/train_model.py

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


@transformer
def train_model(df, *args, **kwargs):
    max_depth = int(kwargs.get("max_depth", 3))

    train_df = df[df["split"] == "train"]
    test_df = df[df["split"] == "test"]

    feature_cols = [
        col for col in df.columns
        if col not in ["target", "split"]
    ]

    X_train = train_df[feature_cols]
    y_train = train_df["target"]

    X_test = test_df[feature_cols]
    y_test = test_df["target"]

    model = DecisionTreeClassifier(
        max_depth=max_depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    test_accuracy = accuracy_score(y_test, predictions)

    return {
        "model": model,
        "test_accuracy": test_accuracy,
        "accuracy": test_accuracy,
        "predictions": predictions,
        "y_test": y_test
    }


@test
def test_output(output, *args):
    assert output is not None
    assert output["model"] is not None
    assert output["test_accuracy"] > 0.80, (
        f"Test accuracy too low: {output['test_accuracy']:.2%}"
    )

Overwriting /content/iris_project/transformers/train_model.py


In [44]:
!rm -f /content/iris_predictions.csv

In [45]:
! /content/mage-env/bin/mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 1}'

DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
FAIL: test_output (block: train_model)
--------------------------------------------------------------
Traceback (most recent call last):
  File "/content/mage-env/lib/python3.12/site-packages/mage_ai/data_preparation/executors/pipeline_executor.py", line 82, in execute
    loop = asyncio.get_running_loop()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: no running event loop

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/mage-env/lib/python3.12/site-packages/mage_ai/data_preparation/models/block/__init__.py", line 3350, in run_tests
    test_function(*outputs)
  File "<string>", line 47, in test_output
AssertionE

In [46]:
!rm -f /content/iris_predictions.csv

In [47]:
! /content/mage-env/bin/mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 2}'

DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [48]:
import os
import pandas as pd

path = "/content/iris_predictions.csv"

print("CSV created:", os.path.exists(path))

if os.path.exists(path):
    df = pd.read_csv(path)
    accuracy = (df["actual"] == df["predicted"]).mean()
    print(f"Test accuracy: {accuracy:.2%}")
    print("Prediction rows:", len(df))

CSV created: True
Test accuracy: 0.00%
Prediction rows: 30


In [49]:
import pandas as pd

df = pd.read_csv("/content/iris_predictions.csv")

display(df.head(10))

print("\nActual values:")
print(df["actual"].value_counts())

print("\nPredicted values:")
print(df["predicted"].value_counts())

print("\nMatching rows:")
print((df["actual"] == df["predicted"]).sum())

,actual,predicted
0,{'target': 0},numpy.ndarray
1,{'target': 0},numpy.ndarray
2,{'target': 0},numpy.ndarray
3,{'target': 0},numpy.ndarray
4,{'target': 0},numpy.ndarray
5,{'target': 0},numpy.ndarray
6,{'target': 0},numpy.ndarray
7,{'target': 0},numpy.ndarray
8,{'target': 0},numpy.ndarray
9,{'target': 0},numpy.ndarray



Actual values:
actual
{'target': 2}    11
{'target': 0}    10
{'target': 1}     9
Name: count, dtype: int64

Predicted values:
predicted
numpy.ndarray    30
Name: count, dtype: int64

Matching rows:
0


In [50]:
%%writefile /content/iris_project/transformers/train_model.py

import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


@transformer
def train_model(df, *args, **kwargs):
    max_depth = int(kwargs.get("max_depth", 3))

    train_df = df[df["split"] == "train"]
    test_df = df[df["split"] == "test"]

    feature_cols = [
        col for col in df.columns
        if col not in ["target", "split"]
    ]

    X_train = train_df[feature_cols]
    y_train = train_df["target"]

    X_test = test_df[feature_cols]
    y_test = test_df["target"]

    model = DecisionTreeClassifier(
        max_depth=max_depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    results = pd.DataFrame({
        "actual": y_test.to_numpy(),
        "predicted": predictions,
        "test_accuracy": accuracy,
        "max_depth": max_depth
    })

    return results


@test
def test_output(output, *args):
    assert output is not None
    assert len(output) == 30

    accuracy = float(output["test_accuracy"].iloc[0])

    assert accuracy > 0.80, (
        f"Test accuracy too low: {accuracy:.2%}"
    )

Overwriting /content/iris_project/transformers/train_model.py


In [51]:
%%writefile /content/iris_project/data_exporters/save_results.py

import os


@data_exporter
def save_results(df, *args, **kwargs):
    output_path = "/content/iris_predictions.csv"

    df.to_csv(output_path, index=False)

    print(f"Predictions saved to: {output_path}")

    return df


@test
def test_output(output, *args):
    assert output is not None
    assert len(output) == 30
    assert "actual" in output.columns
    assert "predicted" in output.columns

Overwriting /content/iris_project/data_exporters/save_results.py


In [52]:
!rm -f /content/iris_predictions.csv

In [53]:
! /content/mage-env/bin/mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 2}'

DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Predictions saved to: /content/iris_predictions.csv
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [54]:
import pandas as pd

df = pd.read_csv("/content/iris_predictions.csv")

display(df.head(10))

accuracy = (df["actual"] == df["predicted"]).mean()

print(f"Test accuracy: {accuracy:.2%}")
print("Rows:", len(df))

,actual,predicted,test_accuracy,max_depth
0,0,0,0.966667,2
1,0,0,0.966667,2
2,0,0,0.966667,2
3,0,0,0.966667,2
4,0,0,0.966667,2
5,0,0,0.966667,2
6,0,0,0.966667,2
7,0,0,0.966667,2
8,0,0,0.966667,2
9,0,0,0.966667,2


Test accuracy: 96.67%
Rows: 30


In [55]:
!rm -f /content/iris_predictions.csv
! /content/mage-env/bin/mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": N}'

DBT library not installed.
Traceback (most recent call last):
  File "/content/mage-env/lib/python3.12/site-packages/mage_ai/cli/utils.py", line 21, in parse_runtime_variables
    vars_parsed = json.loads(variables_str)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/.local/share/uv/python/cpython-3.12.16-linux-x86_64-gnu/lib/python3.12/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/.local/share/uv/python/cpython-3.12.16-linux-x86_64-gnu/lib/python3.12/json/decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/.local/share/uv/python/cpython-3.12.16-linux-x86_64-gnu/lib/python3.12/json/decoder.py", line 356, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 15 (char 14)
INFO:mage_ai.data_preparation.execut

In [56]:
!rm -f /content/iris_predictions.csv

! /content/mage-env/bin/mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 3}'

DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Predictions saved to: /content/iris_predictions.csv
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [57]:
import os
import pandas as pd

path = "/content/iris_predictions.csv"

print("CSV created:", os.path.exists(path))

if os.path.exists(path):
    df = pd.read_csv(path)
    print("Max depth:", df["max_depth"].iloc[0])
    print("Test accuracy:", f'{df["test_accuracy"].iloc[0]:.2%}')
    print("Rows:", len(df))

CSV created: True
Max depth: 3
Test accuracy: 100.00%
Rows: 30


In [58]:
!rm -f /content/iris_predictions.csv

! /content/mage-env/bin/mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 4}'

DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Predictions saved to: /content/iris_predictions.csv
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [59]:
import os
import pandas as pd

path = "/content/iris_predictions.csv"
print("CSV created:", os.path.exists(path))

if os.path.exists(path):
    df = pd.read_csv(path)
    print("Max depth:", df["max_depth"].iloc[0])
    print("Test accuracy:", f'{df["test_accuracy"].iloc[0]:.2%}')
    print("Rows:", len(df))

CSV created: True
Max depth: 4
Test accuracy: 100.00%
Rows: 30


In [60]:
!rm -f /content/iris_predictions.csv

! /content/mage-env/bin/mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 5}'

DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Predictions saved to: /content/iris_predictions.csv
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [61]:
import os
import pandas as pd

path = "/content/iris_predictions.csv"
print("CSV created:", os.path.exists(path))

if os.path.exists(path):
    df = pd.read_csv(path)
    print("Max depth:", df["max_depth"].iloc[0])
    print("Test accuracy:", f'{df["test_accuracy"].iloc[0]:.2%}')
    print("Rows:", len(df))

CSV created: True
Max depth: 5
Test accuracy: 100.00%
Rows: 30


C3 train

In [62]:

%%writefile /content/iris_project/transformers/evaluate_model.py

import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report


@transformer
def evaluate_model(df, *args, **kwargs):
    actual = df["actual"]
    predicted = df["predicted"]

    # Generate confusion matrix
    cm = confusion_matrix(actual, predicted, labels=[0, 1, 2])

    cm_df = pd.DataFrame(
        cm,
        index=["Actual 0", "Actual 1", "Actual 2"],
        columns=["Predicted 0", "Predicted 1", "Predicted 2"]
    )

    # Generate classification metrics
    report = classification_report(
        actual,
        predicted,
        labels=[0, 1, 2],
        output_dict=True,
        zero_division=0
    )

    metrics_df = pd.DataFrame(report).T
    metrics_df = metrics_df.loc[
        ["0", "1", "2", "macro avg", "weighted avg"],
        ["precision", "recall", "f1-score", "support"]
    ]

    print("CONFUSION MATRIX")
    print(cm_df)

    print("\nCLASSIFICATION METRICS")
    print(metrics_df)

    return {
        "confusion_matrix": cm_df,
        "metrics": metrics_df
    }


@test
def test_output(output, *args):
    assert output is not None
    assert output["confusion_matrix"].shape == (3, 3)
    assert len(output["metrics"]) == 5
    assert output["confusion_matrix"].values.sum() == 30


Writing /content/iris_project/transformers/evaluate_model.py


In [63]:

%%writefile /content/iris_project/pipelines/iris_classifier/metadata.yaml

name: iris_classifier
type: python

blocks:
  - uuid: load_iris
    name: load_iris
    type: data_loader
    upstream_blocks: []
    downstream_blocks:
      - prepare_data

  - uuid: prepare_data
    name: prepare_data
    type: transformer
    upstream_blocks:
      - load_iris
    downstream_blocks:
      - train_model

  - uuid: train_model
    name: train_model
    type: transformer
    upstream_blocks:
      - prepare_data
    downstream_blocks:
      - save_results
      - evaluate_model

  - uuid: save_results
    name: save_results
    type: data_exporter
    upstream_blocks:
      - train_model
    downstream_blocks: []

  - uuid: evaluate_model
    name: evaluate_model
    type: transformer
    upstream_blocks:
      - train_model
    downstream_blocks: []


Overwriting /content/iris_project/pipelines/iris_classifier/metadata.yaml


In [64]:
! /content/mage-env/bin/mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 3}'

DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Predictions saved to: /content/iris_predictions.csv
CONFUSION MATRIX
          Predicted 0  Predicted 1  Predicted 2
Actual 0           10            0            0
Actual 1            0            9            0
Actual 2            0            0           11

CLASSIFICATION METRICS
              precision  recall  f1-score  support
0                   1.0     1.0       1.0     10.0
1                   1.0     1.0       1.0      9.0
2                   1.0     1.0       1.0     11.0
macro avg           1.0     1.0       1.0     30.0
weighted avg        1.0     1.0       1.0     30.0
---------------------------

C4 start


In [65]:


%%writefile /content/iris_project/transformers/validate_iris.py

@transformer
def validate_iris(df, *args, **kwargs):
    # Check for missing values
    assert df.isnull().sum().sum() == 0, "Dataset contains missing values"

    # Check dataset size
    assert len(df) == 150, "Expected exactly 150 rows"

    # Check number of target classes
    assert df["target"].nunique() == 3, "Expected exactly 3 classes"

    print("Iris dataset validation successful!")
    return df


@test
def test_output(output, *args):
    assert output is not None
    assert len(output) == 150
    assert output.isnull().sum().sum() == 0
    assert output["target"].nunique() == 3


Writing /content/iris_project/transformers/validate_iris.py


In [66]:

%%writefile /content/iris_project/pipelines/iris_classifier/metadata.yaml

name: iris_classifier
type: python

blocks:
  - uuid: load_iris
    name: load_iris
    type: data_loader
    upstream_blocks: []
    downstream_blocks:
      - validate_iris

  - uuid: validate_iris
    name: validate_iris
    type: transformer
    upstream_blocks:
      - load_iris
    downstream_blocks:
      - prepare_data

  - uuid: prepare_data
    name: prepare_data
    type: transformer
    upstream_blocks:
      - validate_iris
    downstream_blocks:
      - train_model

  - uuid: train_model
    name: train_model
    type: transformer
    upstream_blocks:
      - prepare_data
    downstream_blocks:
      - save_results
      - evaluate_model

  - uuid: save_results
    name: save_results
    type: data_exporter
    upstream_blocks:
      - train_model
    downstream_blocks: []

  - uuid: evaluate_model
    name: evaluate_model
    type: transformer
    upstream_blocks:
      - train_model
    downstream_blocks: []


Overwriting /content/iris_project/pipelines/iris_classifier/metadata.yaml


In [67]:
! /content/mage-env/bin/mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 3}'

DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
Iris dataset validation successful!
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Predictions saved to: /content/iris_predictions.csv
CONFUSION MATRIX
          Predicted 0  Predicted 1  Predicted 2
Actual 0           10            0            0
Actual 1            0            9            0
Actual 2            0            0           11

CLASSIFICATION METRICS
              precision  recall  f1-score  support
0                   1.0     1.0       1.0     10.0
1                   1.0     1.0       1.0      9.0
2                   1.0     1.0       1.0     11.0
macro avg   

In [68]:
!cd /content && zip -r iris_project.zip iris_project

  adding: iris_project/ (stored 0%)
  adding: iris_project/custom/ (stored 0%)
  adding: iris_project/custom/__init__.py (stored 0%)
  adding: iris_project/data_loaders/ (stored 0%)
  adding: iris_project/data_loaders/__pycache__/ (stored 0%)
  adding: iris_project/data_loaders/__pycache__/__init__.cpython-313.pyc (deflated 21%)
  adding: iris_project/data_loaders/__pycache__/load_iris.cpython-313.pyc (deflated 36%)
  adding: iris_project/data_loaders/.ipynb_checkpoints/ (stored 0%)
  adding: iris_project/data_loaders/load_iris.py (deflated 45%)
  adding: iris_project/data_loaders/__init__.py (stored 0%)
  adding: iris_project/metadata.yaml (deflated 52%)
  adding: iris_project/utils/ (stored 0%)
  adding: iris_project/utils/__init__.py (stored 0%)
  adding: iris_project/dbt/ (stored 0%)
  adding: iris_project/dbt/profiles.yml (deflated 20%)
  adding: iris_project/io_config.yaml (deflated 62%)
  adding: iris_project/charts/ (stored 0%)
  adding: iris_project/charts/__init__.py (stored 

In [69]:
from google.colab import files
files.download('/content/iris_project.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>